<a href="https://colab.research.google.com/github/tikent38/Learn_Quantum_Computing_Notebooks/blob/main/Jupyter_Notebooks/Bernstein_Vazirani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Initial Set Up

import ipywidgets as widgets
from IPython.display import display

def ask_mcq(qid, question, options, correct):
    radio = widgets.RadioButtons(
        options=options,
        layout=widgets.Layout(width='auto'),
        style={'description_width': '0px'}
    )

    radio.add_class("big-radio")

    btn = widgets.Button(description="Check answer", button_style='primary')
    out = widgets.Output()

    def check_answer(b):
        with out:
            out.clear_output()

            if radio.value == correct:
                print("✅ Correct!")
            else:
                print(f"❌ Incorrect, try again")

    btn.on_click(check_answer)

    display(widgets.HTML(f"<b style='font-size:24px'>{question}</b>"))
    display(radio, btn, out)

    display(widgets.HTML("""
    <style>
    .big-radio label {
        font-size: 20px !important;
    }
    </style>
    """))



#Bernstein-Vazirani Algorithm

## Problem Statement

Assume you are given a bit string $s$. You don't know what the string is, only that it contains bits. You are also given a function $f(x)$ that you can query which will tell you if the bits match a pattern. $f(x)$ is defined as $( s \cdot x )\bmod 2$ where $s \cdot x $ is the string dot product between $s$ and $x$. Written out $f(x) = x \cdot s = x_{1}s_{1} \oplus x_{2}s_{2} \oplus \cdots \oplus x_{n}s_{n}$.

For example, if $s = 101$ then for the follwing "$x$"s $f(x)$ evaluates to:



*   $x = 000$, $f(x) = (1\cdot0)\oplus(0\cdot0)\oplus(1\cdot0)\bmod 2 = 0$
*   $x = 001$, $f(x) = (1\cdot0)\oplus(0\cdot0)\oplus(1\cdot1)\bmod 2 = 1$
*   $x = 010$, $f(x) = (1\cdot0)\oplus(0\cdot1)\oplus(1\cdot0)\bmod 2 = 0$
*   $x = 011$, $f(x) = (1\cdot0)\oplus(0\cdot1)\oplus(1\cdot1)\bmod 2 = 1$
*   $x = 111$, $f(x) = (1\cdot1)\oplus(0\cdot1)\oplus(1\cdot1)\bmod 2 = 0$

and so on.

The goal is to find what the bits the *s* string contains using the least amount of queries to the function.

## Classical Solution
The classical solution to the problem is to go 1 by 1 checking each bit in $s$, this has a runtime of $O(N)$.


For example if $s = 1011$ we can query the functions with the following $x$ inputs:

*   $x = 1000$, $f(x) = (1\cdot1)\oplus(0\cdot0)\oplus(1\cdot0)\oplus(1\cdot0)\bmod 2 = 1$
*   $x = 0100$, $f(x) = (1\cdot0)\oplus(0\cdot1)\oplus(1\cdot0)\oplus(1\cdot0)\bmod 2 = 0$
*   $x = 0010$, $f(x) = (1\cdot0)\oplus(0\cdot0)\oplus(1\cdot1)\oplus(1\cdot0)\bmod 2 = 1$
*   $x = 0001$, $f(x) = (1\cdot0)\oplus(0\cdot0)\oplus(1\cdot0)\oplus(1\cdot1)\bmod 2 = 1$

Solving this classically requires we query the function once with a $1$ shifting right each time. This means we need $n$ queries when $n$ is the length of the string, hence the $O(N)$ runtime.




In [ ]:
# @title Question 1
ask_mcq(
    "q1",
    "Given the follwing x values and outputs to f(x), what must 's' be <br> x = 1000 f(x) = 0  <br> x = 0100 f(x) = 0  <br> x = 0010 f(x) = 1  <br> x = 0001 f(x) = 0",
    ["1101", "0010", "0000","1001"],
    correct="0010"
)



HTML(value="<b style='font-size:24px'>Given the follwing x values and outputs to f(x), what must 's' be <br> x…

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('1101', '0010', '0000', '1001'…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 2
ask_mcq(
    "q2",
    "If s = 1101 and x = 1011, what is f(x) = (s · x) mod 2 ?",
    ["0", "1", "2", "3"],
    correct="0"
)

HTML(value="<b style='font-size:24px'>If s = 1101 and x = 1011, what is f(x) = (s · x) mod 2 ?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('0', '1', '2', '3'), style=Des…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

## Quantum Solution
The quantum solution to this problem takes advantage of superposition. This allows us to query the function with all possible values of $x$ at once, and encode the values of the function in the phase of each state. Then we can use interference to change the amplitudes so that the only amplitude left as 1 is amplitude of the basis state corresponding to $s$ and the amplitdues of the rest of the states become 0.

This is done by preparing the circuit with $n$ qubits starting in the 0 state and one extra *ancilla* qubit. The ancilla is used to encode the phase and is prepared to be in the $|-\rangle$ state where $|-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt2}$. Lets look at how to prepare the ancilla to be in this state. For this we will need to utilize a new gate, the $Z$ gate. The $Z$ gate is defined as the following:


*   $Z|0\rangle = |0\rangle$
*   $Z|1\rangle = -|1\rangle$

This means applying $Z$ to $|0\rangle$ does nothing while applying it to $|1\rangle$ flips the phase.

Now we have everything we need to transform $|0\rangle -> |-\rangle$. First we start by applying the Hadamard $H$ gate. $H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt2}$. This is close to what we need, but the phase of the $|1\rangle$ state is wrong, this is where the $Z$ gate comes in. We want to flip the phase of $|1\rangle$ while leaving  $|0\rangle$ completely alone. This is exactly what the $Z$ gate does. $Z\frac{|0\rangle + |1\rangle}{\sqrt2} = \frac{|0\rangle - |1\rangle}{\sqrt2}$ which is equal to $|-\rangle$. We have now successfully prepared the ancilla qubit.

###Why the ancilla qubit?
Ancilla qubits are at their core just regular qubits, but they serve a special function in certain quantum algorithms. In general they are often used as helper qubits and can store temporary information about the system without interfering with the main input qubits.

For us specifically, we want a way to keep track of the result $f(x)=( s \cdot x )\bmod 2$. We will do this through the use of $CNOT$ gates and encoding it in the ancilla qubit's phase. Lets first examine why we want to prepare the ancilla qubit in the $|-\rangle$ state. Try applying a $X$ or $NOT$ gate to the following basis states and check your answers below:



1.   $X|0\rangle$
2.   $X|1\rangle$
3.   $X|+\rangle$
4.   $X|-\rangle$

Where $|+\rangle$ comes from applying the Hadamard gate to the $|0\rangle$ state and $|-\rangle$ comes from applying the Hadamard gate to the $|1\rangle$ state.
$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt2}=|+\rangle \qquad H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt2}=|-\rangle$$






<details>
  <summary>Check your answer<br></summary>



1.   $X|0\rangle = |1\rangle$
2.   $X|1\rangle = |0\rangle$
3.   $X|+\rangle =  \frac{X|0\rangle + X|1\rangle}{\sqrt2}=\frac{|1\rangle + |0\rangle}{\sqrt2}=\frac{|0\rangle + |1\rangle}{\sqrt2}=|+\rangle$
4.   $X|-\rangle =  \frac{X|0\rangle - X|1\rangle}{\sqrt2}=\frac{|1\rangle - |0\rangle}{\sqrt2}=-\frac{|0\rangle - |1\rangle}{\sqrt2}=-|-\rangle$



</details>

In [ ]:
# @title Question 3
ask_mcq(
    "q3",
    "In Qiskit, which qubit is |100⟩ referring to being 1?",
    [
        "qubit 0",
        "qubit 1",
        "qubit 2",
        "all qubits"
    ],
    correct="qubit 2"
)

HTML(value="<b style='font-size:24px'>In Qiskit, which qubit is |100⟩ referring to being 1?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('qubit 0', 'qubit 1', 'qubit 2…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

If the goal is to keep track of how many times a $NOT$ gate is applied$\bmod2$, hopefully you can see now why we need the ancilla qubit to be in the $|-\rangle$ state. If its in the $|0\rangle$ or $|1\rangle$ state applying the $NOT$ gate just flips which basis state its in which isnt helpful for encoding information in the phase. Applying it to the $|+\rangle$ state changes nothing however when applied to the $|-\rangle$ state, the phase is flipped. If the amplitude is negative then it means the phase has been flipped an odd amount of times so ($NOT$ applications $\bmod2) = 1$. If the amplitude is positive then it means the phase has been flipped an even amount of times so ($NOT$ applications $\bmod2) = 0$.

### The Oracle
The oracle applies a $CNOT$ gate for each qubit where the corresponding bit in $s$ is $1$. The control qubit is an input qubit and the target is the ancilla qubit. For example if $s = 101$ and we were to apply it to the $|011\rangle$ basis state it will do the following:


*   0th qubit, $s = 1$ so apply a $CNOT$ gate, the basis qubit is $1$ so $NOT$ gate is applied
*   1st qubit, $s = 0$ so do not apply a $CNOT$ gate
*   2nd qubit, $s = 1$ so apply a $CNOT$ gate, the basis qubit is $0$ so $NOT$ gate is not applied

### Full Algorithm
We start by preparing a system with $n$ qubits in the $|0\rangle$ state and a ($n+1$)th qubit in the $|-\rangle$ state. Then we apply a $H$ gate to the $n$ input qubits to put them in superposition. Then we apply the oracle to the system to change the phase of certain basis states. Then we apply a $H$ gate to all the input qubits again to cause them to interfere with eachother condensing the state of the system into a singular basis state with amplitude of $1$
STEPS


1.   Prepare inital system
2.   Apply $H$ gates to input qubits
3.   Apply Oracle to input qubits
4.   Apply $H$ gates to input qubits
5.   Measure system




## Mathmatical walkthrough
Lets say we are given the string $s = 110$

**Step 1: Prepare the intial system**

The system is prepared to be in the state $|000\rangle \otimes |-\rangle$

* The reason why we keep the $|-\rangle$ tensored in a seperate ket will become clear later.

**Step 2: Apply $H$ gate to input qubits**

This will give us the state
$$\frac{1}{\sqrt8}(|000\rangle+|001\rangle+|010\rangle+|011\rangle+|100\rangle+|101\rangle+|110\rangle+|111\rangle)\otimes |-\rangle $$

**Step 3: Apply the Oracle to the input qubits**

For this it, is easier to write the $|-\rangle$ state tensored with each individual basis state


| Basis state tensor $|-\rangle$ | $S$ | Applied $CNOT$ qubits | End basis tensor $|-\rangle$ |
|--------------------------------|-----|-----------------------|-----------|
| $|000\rangle\otimes|-\rangle$  | 110 | $None$|$|000\rangle\otimes|-\rangle$|
| $|001\rangle\otimes|-\rangle$  | 110 | $None$|$|001\rangle\otimes|-\rangle$|
| $|010\rangle\otimes|-\rangle$  | 110 | $1st$|$|010\rangle\otimes-|-\rangle$|
| $|011\rangle\otimes|-\rangle$  | 110 | $1st$|$|011\rangle\otimes-|-\rangle$|
| $|100\rangle\otimes|-\rangle$  | 110 | $2nd$|$|100\rangle\otimes-|-\rangle$|
| $|101\rangle\otimes|-\rangle$  | 110 | $2nd$|$|101\rangle\otimes-|-\rangle$|
| $|110\rangle\otimes|-\rangle$  | 110 | $2nd,1st$|$|110\rangle\otimes|-\rangle$|
| $|111\rangle\otimes|-\rangle$  | 110 | $2nd,1st$|$|111\rangle\otimes|-\rangle$|

The table shows how the oracle changes each of the basis states, specifially which basis states get their phase flipped for $S = 110$

This state of the system after the oracle is applied is:
$$\frac{1}{\sqrt8}(|000\rangle+|001\rangle-|010\rangle-|011\rangle-|100\rangle-|101\rangle+|110\rangle+|111\rangle)\otimes |-\rangle $$

**Step 4: Apply $H$ gate to input qubits**

Now we will apply Hadamard gates to cause the states to interfere and condense the states into one basis state. Lets walk through how applying the $H$ gates changes the system one at a time.

*Inital state*
$$\frac{1}{\sqrt8}(|000\rangle+|001\rangle-|010\rangle-|011\rangle-|100\rangle-|101\rangle+|110\rangle+|111\rangle)\otimes |-\rangle $$
*$H$ on the 2nd qubit*

lets move the terms around to make it easier to see how we compute this

*Inital state rewritten*
$$\frac{1}{\sqrt8}(|000\rangle-|100\rangle+|001\rangle-|101\rangle+|110\rangle-|010\rangle-|011\rangle+|111\rangle)\otimes |-\rangle $$

We know that when we apply a $H$ gate to a qubit twice, we get the orignal state back. Also remember that $H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt2}$ and that $H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt2}$

Lets look specifically at the $|000\rangle-|100\rangle$ terms. If we look at the 2nd qubit (leftmost qubit) we see that it looks like $|0\rangle - |1\rangle$ with the other two qubits being $|00\rangle$. Imagine we start with the state $|100\rangle$ and then apply a $H$ gate to the 2nd qubit we get the result $|000\rangle-|100\rangle$. Ignoring coefficents these are the same basis states as we were looking at above, we also know if we apply a $H$ gate again we get the original $|100\rangle$ state back.
$$\frac{1}{\sqrt8}(|000\rangle-|100\rangle+|001\rangle-|101\rangle+|110\rangle-|010\rangle-|011\rangle+|111\rangle)\otimes |-\rangle $$

Going back to our full quantum system, lets apply the idea we just talked about.

Applying a $H$ gate to the 2nd qubit causes the basis states to interfere with eachother and after combining state we get the following system:
$$\frac{1}{\sqrt4}(|100\rangle+|101\rangle-|110\rangle-|111\rangle)\otimes |-\rangle $$

Now lets apply a $H$ gate to the 0th qubit, we get the system:
$$\frac{1}{\sqrt2}(|100\rangle-|110\rangle)\otimes |-\rangle $$

And finally apply the last $H$ gate to the 1st qubit:
$$\frac{1}{\sqrt1}(|110\rangle)\otimes |-\rangle $$

**Step 5: Measure the system**
Our system now can be written as:
$$\frac{1}{\sqrt1}(|110\rangle)\otimes |-\rangle $$
When we measure the input qubits we see that we will measure the $|110\rangle$ state with 100% probabilty as its the only basis state left. We can also see $110$ is also the original bit string $S$. Using the Bernstein-Vazirani Algorithm we have shown how to use ideas from quantum mechanics to determine what a random string $S$ must be.

**Your turn**

Try running throught the algorithm for $S=001$. Write out the system and the basis states at each step.

<details>
  <summary>Check your answer<br></summary>



1.   **Prepare the inital system**
$$|000\rangle \otimes |-\rangle$$
2.   **Apply $H$ gates to input qubits**
$$\frac{1}{\sqrt8}(|000\rangle+|001\rangle+|010\rangle+|011\rangle+|100\rangle+|101\rangle+|110\rangle+|111\rangle)\otimes |-\rangle $$
3. **Apply Oracle to input qubits**
$$\frac{1}{\sqrt8}(|000\rangle-|001\rangle+|010\rangle-|011\rangle+|100\rangle-|101\rangle+|110\rangle-|111\rangle)\otimes |-\rangle $$
4. **Apply $H$ gates to input qubits**
$$\frac{1}{\sqrt8}(|000\rangle+|100\rangle-|001\rangle-|101\rangle+|010+|110\rangle\rangle-|011\rangle-|111\rangle)\otimes |-\rangle $$
$$\frac{1}{\sqrt4}(|000\rangle-|001\rangle+|010\rangle-|011\rangle)\otimes |-\rangle $$
$$\frac{1}{\sqrt2}(|001\rangle+|011\rangle)\otimes |-\rangle $$
$$\frac{1}{\sqrt1}(|001\rangle)\otimes |-\rangle $$
**Step 5: Measure the system**

$$1\,|001\rangle$$
</details>

In [ ]:
# @title Question 4
ask_mcq(
    "q4",
    "Why is the ancilla prepared in the |−⟩ state?",
    [
        "To store f(x) as a classical bit",
        "Because X|−⟩ = −|−⟩ which encodes phase information",
        "To increase entanglement only",
        "Because Hadamard gates require it"
    ],
    correct="Because X|−⟩ = −|−⟩ which encodes phase information"
)

HTML(value="<b style='font-size:24px'>Why is the ancilla prepared in the |−⟩ state?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('To store f(x) as a classical …

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 5
ask_mcq(
    "q5",
    "If the ancilla is prepared in |+⟩ instead of |−⟩, what happens?",
    [
        "Phase kickback still occurs",
        "The oracle encodes nothing useful in phase",
        "The circuit measures faster",
        "The answer flips"
    ],
    correct="The oracle encodes nothing useful in phase"
)

HTML(value="<b style='font-size:24px'>If the ancilla is prepared in |+⟩ instead of |−⟩, what happens?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('Phase kickback still occurs',…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 6
ask_mcq(
    "q6",
    "How is the Bernstein–Vazirani oracle implemented in the circuit?",
    [
        "Apply Z gates to every qubit",
        "Apply CNOTs from input qubits to the ancilla where s_i = 1",
        "Measure after each gate",
        "Swap all qubits"
    ],
    correct="Apply CNOTs from input qubits to the ancilla where s_i = 1"
)

HTML(value="<b style='font-size:24px'>How is the Bernstein–Vazirani oracle implemented in the circuit?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('Apply Z gates to every qubit'…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 7
ask_mcq(
    "q7",
    "In Qiskit (little-endian ordering), if s = 1010, which qubits should control CNOTs in the oracle?",
    [
        "qubits 0 and 2",
        "qubits 1 and 3",
        "all qubits",
        "no qubits"
    ],
    correct="qubits 1 and 3"
)

HTML(value="<b style='font-size:24px'>In Qiskit (little-endian ordering), if s = 1010, which qubits should con…

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('qubits 0 and 2', 'qubits 1 an…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 8
ask_mcq(
    "q8",
    "After the full algorithm, measuring the input qubits return:",
    [
        "a random bit string",
        "the parity of s",
        "the secret string s with probability 1",
        "all zeros"
    ],
    correct="the secret string s with probability 1"
)

HTML(value="<b style='font-size:24px'>After the full algorithm, measuring the input qubits return:</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('a random bit string', 'the pa…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 9
ask_mcq(
    "q9",
    "What is the purpose of the second layer of Hadamard gates?",
    [
        "To create superposition",
        "To measure the qubits",
        "To cause interference that concentrates amplitude on |s⟩",
        "To reset the ancilla"
    ],
    correct="To cause interference that concentrates amplitude on |s⟩"
)

HTML(value="<b style='font-size:24px'>What is the purpose of the second layer of Hadamard gates?</b>")

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('To create superposition', 'To…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 10
ask_mcq(
    "q10",
    "Classically, how many oracle queries are required in the worst case to determine an n-bit secret string s?",
    ["1", "log(n)", "n", "2^n"],
    correct="n"
)

HTML(value="<b style='font-size:24px'>Classically, how many oracle queries are required in the worst case to d…

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('1', 'log(n)', 'n', '2^n'), st…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

In [ ]:
# @title Question 11
ask_mcq(
    "q11",
    "The time complexity of Bernstein–Vazirani in terms of oracle calls is:",
    ["O(n)", "O(log n)", "O(1)", "O(2^n)"],
    correct="O(1)"
)

HTML(value="<b style='font-size:24px'>The time complexity of Bernstein–Vazirani in terms of oracle calls is:</…

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('O(n)', 'O(log n)', 'O(1)', 'O…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …



###Quantum Circuit
We start by creating a circuit with $n+1$ qubits. Then, we apply a $H$ gate and then a $Z$ gate to the $n+1$th qubit to prepare it in the $|-\rangle $ state. Then we apply $H$ gates to each of the input qubits to put them in superposition. Then we will define and apply the Oracle. Then we apply a $H$ gate to eadh of the input qubits again. Finally we measure the input qubits.

The code implementation for this is shown below





In [ ]:
# @title Qiskit imports
#run this if you have never installed qiskit before
!pip install -q qiskit
!pip install -q qiskit-aer
!pip install -q pylatexenc
import qiskit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_state_city
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector
import numpy as np
%matplotlib inline

#CODE TO GENERATE RANDOM STRINGS
n = 3 #LENGTH OF STRING
s = ''.join(np.random.choice(['0','1'], size=n))
print("Secret string:", s)


### FILL IN THE CODE FOR run_bv
def run_bv(s, shots="1024"):

  n = len(s)
  s = s[::-1]  # reverse string for little-endian convention

  ### TODO CREATE THE QUANTUM CIRCUIT WITH n CLASSICAL BITS AND n+1 QUANTUM BITS
  qc =

  ### TODO PREPARE THE INITIAL STATE OF THE ANCILLA QUBIT


  def Oracle(circuit, s):
      for i, bit in enumerate(s):
          if bit == '1':
              ### TODO FINISH ORACLE CODE


  ### TODO PUT INPUT QUBITS IN SUPERPOSITION

  ### TODO APPLY ORACLE TO INPUT QUBITS


  ## TODO APPLY H GATES TO INPUT QUBITS


  #Statevector
  sv = Statevector.from_instruction(qc)

  qc.measure(range(n), range(n))
  #use AerSimulator
  sim = AerSimulator()
  compiled = transpile(qc, sim)
  #run Simulation
  result = sim.run(compiled, shots=1024).result()
  #extract counts
  counts = result.get_counts()
  print(counts)
  qc.draw(output="mpl")
  return(counts)




counts = run_bv(s)

plot_histogram(counts)
plt.show()

Secret string: 111
{'111': 1024}


In [ ]:
# @title Hidden Test Cells
import numpy as np

def run_all_tests():
    # ---- fixed tests ----
    test_strings = ["000", "111", "101", "010", "001"]

    for s in test_strings:
        counts = run_bv(s)
        measured = max(counts, key=counts.get)
        assert measured == s, f"FAILED fixed test for {s}. Measured = {measured}"

    # ---- random tests ----
    for _ in range(10):
        n = 4
        s = ''.join(np.random.choice(['0','1'], size=n))
        counts = run_bv(s)
        measured = max(counts, key=counts.get)
        assert measured == s, f"FAILED random test for {s}"

    print("✅ All tests passed!")


run_all_tests()

{'000': 1024}
{'111': 1024}
{'101': 1024}
{'010': 1024}
{'001': 1024}
{'1010': 1024}
{'0101': 1024}
{'1101': 1024}
{'1010': 1024}
{'0001': 1024}
{'1001': 1024}
{'1111': 1024}
{'0001': 1024}
{'1011': 1024}
{'0111': 1024}
✅ All tests passed!
